# Analyse BigQuery — TransLog

**Objectif :** exploiter les données du pipeline pour préparer le dashboard Data Studio.

**Source :** tables `translog-pipeline.logistics.fuel_prices` et `weather_daily`.

**Données :** ~30 000 lignes (9 817 stations × 5 carburants).

In [27]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery
from google.oauth2 import service_account

In [28]:
# Configuration
PROJECT_ID = "translog-pipeline"
DATASET = "logistics"

In [29]:
# Chemin vers le fichier de service account
SA_PATH = r"C:\Users\Utilisateur\Documents\Data_Analyst_Formation\Securite\Translog\translog-pipeline-32a828df08d3.json"

In [30]:
credentials = service_account.Credentials.from_service_account_file(SA_PATH)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

print("Client BigQuery initialise :", client.project)

Client BigQuery initialise : translog-pipeline


## 1. Prix moyen par carburant en France

Vue d'ensemble des prix sur l'ensemble des stations.

In [31]:
query1 = f"""
SELECT
  carburant,
  COUNT(*) AS nb_releves,
  ROUND(AVG(prix), 3) AS prix_moyen,
  ROUND(MIN(prix), 3) AS prix_min,
  ROUND(MAX(prix), 3) AS prix_max
FROM `{PROJECT_ID}.{DATASET}.fuel_prices`
GROUP BY carburant
ORDER BY carburant
"""
df_prix = client.query(query1).to_dataframe()
df_prix

c:\Users\Utilisateur\Documents\Data_Analyst_Formation\Big Data\TransLog_Api_Big_Query\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,carburant,nb_releves,prix_moyen,prix_min,prix_max
0,E10,7280,2.170,1.880,2.679
1,E85,3721,0.916,0.769,2.259
2,Gazole,9204,2.358,2.090,2.900
3,SP95,2758,2.226,1.849,2.740
4,SP98,7240,2.275,1.980,2.880


## 2. Top 10 des départements les moins chers pour le Gazole

Filtre : au moins 10 stations par département pour être représentatif.

In [37]:
query2 = f"""
SELECT
  departement,
  COUNT(DISTINCT id_station) AS nb_stations,
  ROUND(AVG(prix), 3) AS prix_moyen
FROM `{PROJECT_ID}.{DATASET}.fuel_prices`
WHERE carburant = 'Gazole'
  AND departement NOT IN ('2A', '2B')
GROUP BY departement
HAVING nb_stations >= 10
ORDER BY prix_moyen ASC, nb_stations DESC
LIMIT 10
"""
df_top10 = client.query(query2).to_dataframe()
df_top10

c:\Users\Utilisateur\Documents\Data_Analyst_Formation\Big Data\TransLog_Api_Big_Query\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,departement,nb_stations,prix_moyen
0,29,142,2.323
1,22,109,2.323
2,44,164,2.325
3,35,141,2.325
4,12,79,2.332
5,85,111,2.334
6,78,137,2.336
7,56,130,2.337
8,64,111,2.338
9,81,66,2.338


**Note méthodologique :** les départements 2A et 2B (Corse) ont été exclus. En raison de leur insularité, les prix y sont structurellement plus élevés et non comparables aux prix continentaux pour un transporteur routier comme TransLog.

**Note méthodologique — Exclusion de la Corse (2A, 2B)**

Les départements 2A (Corse-du-Sud) et 2B (Haute-Corse) ont été exclus de l'analyse comparative principale pour trois raisons :

1. **Insularité** : le carburant y est importé par voie maritime, ce qui ajoute un coût structurel.
2. **Marché atypique** : peu de stations, prix non comparables au continent.
3. **Activité marginale pour TransLog** : la Corse représente une part très faible des tournées.

Ils sont toutefois analysés séparément ci-dessous pour information.

In [38]:
query_corse = f"""
SELECT
  departement,
  COUNT(DISTINCT id_station) AS nb_stations,
  ROUND(AVG(prix), 3) AS prix_moyen,
  ROUND(MIN(prix), 3) AS prix_min,
  ROUND(MAX(prix), 3) AS prix_max
FROM `{PROJECT_ID}.{DATASET}.fuel_prices`
WHERE carburant = 'Gazole'
  AND departement IN ('2A', '2B')
GROUP BY departement
ORDER BY departement
"""
df_corse = client.query(query_corse).to_dataframe()

# Comparaison avec le continent
prix_continent = client.query(f"""
SELECT ROUND(AVG(prix), 3) AS prix_moyen
FROM `{PROJECT_ID}.{DATASET}.fuel_prices`
WHERE carburant = 'Gazole'
  AND departement NOT IN ('2A', '2B')
""").to_dataframe()["prix_moyen"].iloc[0]

print(f"Prix moyen Corse (2A + 2B) : {df_corse['prix_moyen'].mean():.3f} EUR/L")
print(f"Prix moyen continent       : {prix_continent:.3f} EUR/L")
print(f"Surcout Corse              : {df_corse['prix_moyen'].mean() - prix_continent:+.3f} EUR/L")
df_corse

c:\Users\Utilisateur\Documents\Data_Analyst_Formation\Big Data\TransLog_Api_Big_Query\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\Utilisateur\Documents\Data_Analyst_Formation\Big Data\TransLog_Api_Big_Query\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Prix moyen Corse (2A + 2B) : 2.277 EUR/L
Prix moyen continent       : 2.360 EUR/L
Surcout Corse              : -0.083 EUR/L


,departement,nb_stations,prix_moyen,prix_min,prix_max
0,2A,59,2.279,2.195,2.625
1,2B,67,2.276,2.190,2.450


## 3. Écart max-min par département (Gazole)

Identifie les zones où la concurrence est la plus forte ou la plus faible.

In [ ]:
query3 = f"""
SELECT
  departement,
  COUNT(*) AS nb_releves,
  ROUND(MIN(prix), 3) AS prix_min,
  ROUND(MAX(prix), 3) AS prix_max,
  ROUND(MAX(prix) - MIN(prix), 3) AS ecart_prix
FROM `{PROJECT_ID}.{DATASET}.fuel_prices`
WHERE carburant = 'Gazole'
GROUP BY departement
HAVING nb_releves >= 10
ORDER BY ecart_prix DESC
LIMIT 10
"""
df_ecart = client.query(query3).to_dataframe()
df_ecart

c:\Users\Utilisateur\Documents\Data_Analyst_Formation\Big Data\TransLog_Api_Big_Query\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,departement,nb_releves,prix_min,prix_max,ecart_prix
0,75,40,2.200,2.900,0.700
1,67,117,2.249,2.876,0.627
2,68,91,2.250,2.877,0.627
3,85,111,2.250,2.829,0.579
4,17,111,2.225,2.789,0.564
5,49,103,2.250,2.809,0.559
6,28,73,2.242,2.789,0.547
7,06,140,2.250,2.790,0.540
8,33,217,2.210,2.750,0.540
9,83,167,2.249,2.780,0.531


## 4. Jours à risque météo sur les hubs logistiques

Détection : pluie > 5 mm ou vent > 50 km/h.

In [34]:
query4 = f"""
SELECT
  w.hub,
  w.date_prevision,
  w.precipitation,
  w.vent_max,
  w.temp_max,
  w.temp_min
FROM `{PROJECT_ID}.{DATASET}.weather_daily` w
WHERE w.date_collecte = (SELECT MAX(date_collecte) FROM `{PROJECT_ID}.{DATASET}.weather_daily`)
  AND (w.precipitation > 5 OR w.vent_max > 50)
ORDER BY w.date_prevision, w.hub
"""
df_risques = client.query(query4).to_dataframe()
df_risques

c:\Users\Utilisateur\Documents\Data_Analyst_Formation\Big Data\TransLog_Api_Big_Query\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,hub,date_prevision,precipitation,vent_max,temp_max,temp_min
0,Marseille,2026-10-19,21.2,30.8,20.7,12.9
1,Paris,2026-10-19,11.7,16.4,14.4,7.8
2,Marseille,2026-10-20,17.2,24.5,19.0,16.8
3,Paris,2026-10-20,11.2,11.4,8.8,7.9


## 5. KPI pour Data Studio

Trois indicateurs clés à afficher dans le dashboard.

In [39]:
# KPI 1 - Prix moyen du Gazole (continent)
prix_gazole = client.query(f"""
SELECT ROUND(AVG(prix), 3) AS p
FROM `{PROJECT_ID}.{DATASET}.fuel_prices`
WHERE carburant = 'Gazole'
  AND departement NOT IN ('2A', '2B')
""").to_dataframe()["p"].iloc[0]
print(f"KPI 1 - Prix moyen du Gazole (continent) : {prix_gazole:.3f} EUR/L")

# KPI 2 - Ecart max-min (continent)
ecart_max = df_ecart[~df_ecart["departement"].isin(["2A", "2B"])]["ecart_prix"].iloc[0]
print(f"KPI 2 - Ecart max dans un departement (continent) : {ecart_max:.3f} EUR/L")

# KPI 3 - Nombre de stations suivies (continent)
nb_stations = client.query(f"""
SELECT COUNT(DISTINCT id_station) AS nb
FROM `{PROJECT_ID}.{DATASET}.fuel_prices`
WHERE departement NOT IN ('2A', '2B')
""").to_dataframe()["nb"].iloc[0]
print(f"KPI 3 - Nombre de stations suivies (continent) : {nb_stations}")

c:\Users\Utilisateur\Documents\Data_Analyst_Formation\Big Data\TransLog_Api_Big_Query\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


KPI 1 - Prix moyen du Gazole (continent) : 2.360 EUR/L
KPI 2 - Ecart max dans un departement (continent) : 0.700 EUR/L
KPI 3 - Nombre de stations suivies (continent) : 9230


c:\Users\Utilisateur\Documents\Data_Analyst_Formation\Big Data\TransLog_Api_Big_Query\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 6. Conclusion métier

**Constat :** le prix du Gazole varie de plusieurs centimes par litre entre les départements.

**Impact :** pour un plein de 10 000 L, l'écart peut représenter plusieurs centaines d'euros.

**Recommandation :** TransLog peut optimiser ses ravitaillements en ciblant les départements les moins chers et planifier les tournées en tenant compte des jours à risque météo.